In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/baleshsingh/acme-handbook/Acme_Handbook.pdf


In [2]:
!pip install -q google-genai


In [4]:
from google import genai


client = genai.Client(api_key="Gemini Api Key")

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="My name is Balesh,Give some jokes to me,25 words"
)
print(response.text)

Hey Balesh! Why don't skeletons fight each other? 
They don't have the guts! 

Also, I told my wife she was drawing her eyebrows too high. She looked surprised.


In [5]:
document = """
Leave Policy: Every employee gets 18 paid leaves per year. Unused leaves up to 10 can be carried forward to the next year.
Work From Home: Employees can work from home 2 days a week with manager approval. WFH requests must be raised one day before.
Working Hours: Office hours are 10 AM to 7 PM, Monday to Friday. Saturday and Sunday are off.
Notice Period: The notice period for all full-time employees is 60 days. It can be reduced to 30 days with HR approval.
Reimbursement: Travel expenses for client visits are reimbursed within 15 days. Bills must be submitted within 7 days of travel.
"""

def make_chunks(text, chunk_size=30, overlap=5):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunk = " ".join(words[start:start + chunk_size])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = make_chunks(document)
print("Total chunks:", len(chunks))
for i, c in enumerate(chunks):
    print(i, "->", c, "\n")

Total chunks: 5
0 -> Leave Policy: Every employee gets 18 paid leaves per year. Unused leaves up to 10 can be carried forward to the next year. Work From Home: Employees can work from 

1 -> Home: Employees can work from home 2 days a week with manager approval. WFH requests must be raised one day before. Working Hours: Office hours are 10 AM to 7 

2 -> are 10 AM to 7 PM, Monday to Friday. Saturday and Sunday are off. Notice Period: The notice period for all full-time employees is 60 days. It can be reduced 

3 -> days. It can be reduced to 30 days with HR approval. Reimbursement: Travel expenses for client visits are reimbursed within 15 days. Bills must be submitted within 7 days of 

4 -> submitted within 7 days of travel. 



In [6]:
import numpy as np

# Step 3: Embedding of chunks
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=chunks
)
chunk_vectors = np.array([e.values for e in result.embeddings])
print("Shape:", chunk_vectors.shape)

# Step 4: Finding similar chunks from embedding
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(question, top_k=2):
    q = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )
    q_vec = np.array(q.embeddings[0].values)
    scores = [cosine(q_vec, v) for v in chunk_vectors]
    best = np.argsort(scores)[::-1][:top_k]
    return [(int(i), round(scores[i], 3), chunks[i]) for i in best]

for item in retrieve("How many days can I work from home?"):
    print(item, "\n")

Shape: (5, 3072)
(1, np.float64(0.736), 'Home: Employees can work from home 2 days a week with manager approval. WFH requests must be raised one day before. Working Hours: Office hours are 10 AM to 7') 

(0, np.float64(0.683), 'Leave Policy: Every employee gets 18 paid leaves per year. Unused leaves up to 10 can be carried forward to the next year. Work From Home: Employees can work from') 



In [7]:
def answer(question, top_k=2):
    hits = retrieve(question, top_k)
    context = "\n\n".join(h[2] for h in hits)

    prompt = f"""You are a company policy assistant.
Answer ONLY using the context below.
If the answer is not in the context, say "I don't know based on the document."
Keep the answer short.

Context:
{context}

Question: {question}
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

print(answer("How many days can I work from home?"))
print(answer("What is the notice period?"))
print(answer("What is the salary of the CEO?"))

Employees can work from home 2 days a week.
The notice period for all full-time employees is 60 days. It can be reduced to 30 days with HR approval.
I don't know based on the document.


In [8]:
!pip install -q pypdf

In [9]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        print(os.path.join(dirname, f))

/kaggle/input/datasets/baleshsingh/acme-handbook/Acme_Handbook.pdf


In [10]:
!pip install -q pypdf
from pypdf import PdfReader

reader = PdfReader("/kaggle/input/datasets/baleshsingh/acme-handbook/Acme_Handbook.pdf")  
document = "\n".join(page.extract_text() or "" for page in reader.pages)
print("Words:", len(document.split()))

Words: 557


In [11]:
chunks = make_chunks(document, chunk_size=200, overlap=30)
print("Chunks:", len(chunks))

vectors = []
for i in range(0, len(chunks), 50):
    batch = chunks[i:i+50]
    res = client.models.embed_content(model="gemini-embedding-001", contents=batch)
    vectors.extend(e.values for e in res.embeddings)

chunk_vectors = np.array(vectors)
print(chunk_vectors.shape)

Chunks: 4
(4, 3072)


In [13]:
questions = [
    "How many sick days do I get?",
    "What is the hotel reimbursement limit in metro cities?",
    "Can I work from home during probation?",
    "Where do I send a POSH complaint?",
    "What is the company's stock option policy?",
]

for q in questions:
    print("Q:", q)
    try:
        print("A:", answer(q))
    except Exception as e:
        print("Error:", e)
    print()

Q: How many sick days do I get?
A: 8 additional paid sick days per year.

Q: What is the hotel reimbursement limit in metro cities?
A: Hotel stay is reimbursed up to Rs 4,000 per night in metro cities.

Q: Can I work from home during probation?
A: Employees in their first 3 months (probation) are not eligible for WFH except in emergencies approved by HR.

Q: Where do I send a POSH complaint?
A: Complaints can be sent to ic@acme-example.com.

Q: What is the company's stock option policy?
A: I don't know based on the document.



In [14]:
print("Document Q&A Bot ready. Type 'quit' to stop.\n")

while True:
    q = input("Ask: ")
    if q.strip().lower() == "quit":
        print("Bye!")
        break
    if not q.strip():
        continue
    try:
        print("Answer:", answer(q), "\n")
    except Exception as e:
        print("Error:", e, "\n")

Document Q&A Bot ready. Type 'quit' to stop.



Ask:  Quit


Bye!
